In [0]:
from pyspark.sql.functions import col, expr, trim

# -----------------------------
# 1. Read data
# -----------------------------
df = spark.read.table("mini_project_catalog.`01_bronze`.customer_survey")

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.dropDuplicates()

# -----------------------------
# 3. Replace invalid values ("-") with null
# -----------------------------
df = df.replace("-", None)

# -----------------------------
# 4. Trim all string columns
# -----------------------------
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# -----------------------------
# 5. Convert timestamp columns (ISO format)
# -----------------------------
timestamp_cols = [
    "survey_sent_date",
    "survey_response_date"
]

for c in timestamp_cols:
    if c in df.columns:
        df = df.withColumn(c, expr(f"to_timestamp({c})"))

# -----------------------------
# 6. Convert boolean column
# -----------------------------
if "responded_flag" in df.columns:
    df = df.withColumn(
        "responded_flag",
        col("responded_flag").cast("boolean")
    )

# -----------------------------
# 7. FIX rating columns (SAFE CAST 🔥)
# -----------------------------
rating_cols = [
    "delivered_on_time_rating",
    "work_quality_rating",
    "cleanliness_rating",
    "communication_rating",
    "overall_satisfaction_rating"
]

for c in rating_cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            expr(f"CAST(try_cast({c} AS DOUBLE) AS INT)")
        )

# -----------------------------
# 8. Business logic for ratings
# -----------------------------
df = df.withColumn(
    "delivered_on_time_rating",
    expr("""
        CASE 
            WHEN responded_flag = true AND delivered_on_time_rating IS NULL THEN 0
            ELSE delivered_on_time_rating
        END
    """)
)

# -----------------------------
# 9. Filter valid records
# -----------------------------
df = df.filter(col("survey_id").isNotNull())

# -----------------------------
# 10. Write to Silver
# -----------------------------
df.write.mode("overwrite").saveAsTable(
    "mini_project_catalog.02_silver.customer_survey"
)

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.customer_survey LIMIT 10;